# 43 · Relate, rules and a count, checked against pandas: who infected whom in Hagelloch, 1861

## What this checks

This check follows a chain of steps built by hand on the 188 measles cases of Hagelloch, Germany, in 1861. It covers
**Relate a column from another table** at release 0.15.0, with the workflow's rules and **Total within groups**.

The chain gives two numbers for each case:

- **The serial interval.** This is the days between the first symptoms (prodrome) of an infector and of the case
  they infected. Relate joins the cases to themselves on `infector`, to bring in each infector's prodrome date. The
  rules then take one date from the other.
- **The secondary cases.** This is how many cases each child infected. Total within groups counts the cases by
  `infector`. Relate brings the counts back on `case_ID`. The children nobody names as an infector are kept as 0, a
  number the practitioner typed in.

These numbers describe how measles spread from child to child. A wrong join would give a child someone else's
infections.

## How it is checked

We compute the same columns with **pandas**, Python's library for tables, which shares no code with Fieldwork.

- **The serial interval.** pandas joins the cases to themselves, infector to `case_ID`, and takes the difference of
  the two dates in days.
- **The secondary cases.** pandas counts the cases by `infector`, over every `case_ID`, with 0 for a case nobody
  names.

The data are the chain as Fieldwork ran it on the 188 cases.

## What should happen

1. **Every serial interval** is the same, in whole days. A case with no recorded infector has none, on either side.
2. **Every count of secondary cases** is the same, and so are their total and the number of zeros.

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the installed packages.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_43_hagelloch_chains import run

report = run(Path.cwd() / "fixtures" / "chains-hagelloch-001.json")
print("agrees:", report["agrees"])
s = report["serialIntervals"]
print(f"Serial intervals: {s['n']}, median {s['median']:g} days, from {s['min']} to {s['max']}")
c = report["secondaryCases"]
print(f"Secondary cases: {c['sum']} in all, {c['zeros']} children infected no one recorded, the most {c['max']}")
print("Distribution:", c["distribution"])
print("Differs:", report["differs"] or "none")

## The result on record

First recorded on 9 November 2026, against Relate release 0.11.0. Rerun on 26 November 2026, against Relate
release 0.15.0: **Agrees**.

- **Serial intervals.** There are 184, one for each case with a recorded infector. The median is 10 days, ranging
  from 7 to 16. Each equals the difference pandas finds.
- **Secondary cases.** They total 184. 126 children infected nobody and count 0. The most is 30, by case 45. Each
  count equals pandas'.

The result agrees in CPython and in the browser kernel. Building the same chain by hand on the canvas gives the same
numbers.

Running the cell above repeats the check on the same data.

## What this does not show

- Who infected whom. The `infector` column is a reconstruction from the 1861 record of the outbreak. Both sides read
  the same reconstruction.
- The reproduction number, R. The mean of the secondary cases, 184 over 188, is not an estimate of it.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
e = report["environment"]
print("pandas  :", e["pandas"])
print("runtime :", e["runtime"], "· Pyodide" if e["pyodide"] else "")